In [ ]:
#importing python libraries for predicting house prices

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import os

# Input data files are available in the read-only "../input/" directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
data.head()

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
print("No of null values per column:")
print(data.isnull().sum()[data.isnull().sum() > 0])

In [ ]:
#Handling missing values
num_cols = data.select_dtypes(include=['float', 'int']).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].mean())

#Categorical columns where NA/None is a valid missing value
#where NA/None is not a meaningful, expected value according to data_description contents
data['Electrical'] = data['Electrical'].fillna(data['Electrical'].mode()[0])
print(data.isnull().sum()[data.isnull().sum() > 0])

All above columns were already expected to have NA/None as a valid value, based on data_description.txt, so these are not treated as missing values.

In [ ]:
#Feature Engineering for numerical features
data['BsmtTotalUsblSF'] = data['TotalBsmtSF'] - data['BsmtUnfSF']
data['Bathrooms'] = data['BsmtFullBath'] + data['FullBath'] + (0.5*data['BsmtHalfBath']) + (0.5*data['HalfBath'])
data['TotalPorchSF'] = data['OpenPorchSF'] + data['EnclosedPorch'] + data['3SsnPorch'] + data['ScreenPorch']

test_data['BsmtTotalUsblSF'] = data['TotalBsmtSF'] - data['BsmtUnfSF']
test_data['Bathrooms'] = test_data['BsmtFullBath'] + test_data['FullBath'] + (0.5*test_data['BsmtHalfBath']) + (0.5*test_data['HalfBath'])
test_data['TotalPorchSF'] = test_data['OpenPorchSF'] + test_data['EnclosedPorch'] + test_data['3SsnPorch'] + test_data['ScreenPorch']

#Columns (already) combined to create features
cols_to_drop = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
               'BsmtFullBath', 'FullBath', 'BsmtHalfBath', 'HalfBath', 'OpenPorchSF', 
                'EnclosedPorch','3SsnPorch','ScreenPorch']

data = data.drop(columns=cols_to_drop)
test_data = test_data.drop(columns=cols_to_drop)

In [ ]:
#Ordinal Encoding for Categorical Features

#Custom mapping
util_mapping = {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4}
func_mapping = {'Typ': 1, 'Min1': 2, 'Min2': 3, 'Mod': 4, 'Maj1': 5, 'Maj2': 6, 'Sev': 7, 'Sal':8}
pv_map = {'Y': 1,'P': 2,'N': 3}

data['NoofUtil'] = data['Utilities'].map(util_mapping)
data['FuncLvl'] = data['Functional'].map(func_mapping)
data['PavedDvType'] = data['PavedDrive'].map(pv_map)

test_data['NoofUtil'] = test_data['Utilities'].map(util_mapping)
test_data['FuncLvl'] = test_data['Functional'].map(func_mapping)
test_data['PavedDvType'] = test_data['PavedDrive'].map(pv_map)

#Using OrdinalEncoder
encode_cols=['MSZoning','Street', 'Alley', 'LotShape', 'LandContour', 'LotConfig',
             'LandSlope', 'Neighborhood','Condition1','Condition2', 'BldgType',
             'HouseStyle','RoofStyle','RoofMatl','Exterior1st','Exterior2nd',
             'MasVnrType','ExterQual','ExterCond','Foundation','BsmtQual','BsmtCond',
             'BsmtExposure','BsmtFinType1','BsmtFinType2','Heating','HeatingQC',
             'CentralAir','Electrical','KitchenQual','FireplaceQu','GarageType',
             'GarageFinish', 'GarageQual', 'GarageCond','PoolQC','Fence',
             'MiscFeature','SaleType','SaleCondition']
ordinal_encoder = OrdinalEncoder(categories=[
    ['A','C','FV','I','RH','RL','RP','RM'],
    ['Grvl','Pave'],
    ['NA','Grvl','Pave'],
    ['Reg','IR1','IR2','IR3'],
    ['Lvl','Bnk','HLS','Low'],
    ['Inside','Corner','CulDSac','FR2','FR3'],
    ['Gtl', 'Mod', 'Sev'],  
    ['Blmngtn','Blueste','BrDale','BrkSide','ClearCr','CollgCr','Crawfor','Edwards','Gilbert','IDOTRR','MeadowV','Mitchel','Names','NoRidge','NPkVill','NridgHt','NWAmes','OldTown','SWISU','Sawyer','SawyerW','Somerst','StoneBr','Timber','Veenker'],
    ['Artery','Feedr','Norm','RRNn','RRAn','PosN','PosA','RRNe','RRAe'],
    ['Artery','Feedr','Norm','RRNn','RRAn','PosN','PosA','RRNe','RRAe'],
    ['1Fam','2FmCon','Duplx','TwnhsE','TwnhsI'],
    ['1Story','1.5Fin','1.5Unf','2Story','2.5Fin','2.5Unf','SFoyer','SLvl'],
    ['Flat','Gable','Gambrel','Hip','Mansard','Shed'],
    ['ClyTile','CompShg','Membran','Metal','Roll','Tar&Grv','WdShake','WdShngl'],
    ['AsbShng','AsphShn','BrkComm','BrkFace','CBlock','CemntBd','HdBoard','ImStucc','MetalSd','Other','Plywood','PreCast','Stone','Stucco','VinylSd','Wd Sdng','WdShing'],
    ['AsbShng','AsphShn','BrkComm','BrkFace','CBlock','CemntBd','HdBoard','ImStucc','MetalSd','Other','Plywood','PreCast','Stone','Stucco','VinylSd','Wd Sdng','WdShing'],
    ['None','Stone','CBlock','BrkFace','BrkCmn'],
    ['Po','Fa','TA','Gd','Ex'],
    ['Po','Fa','TA','Gd','Ex'],
    ['BrkTil','CBlock','PConc','Slab','Stone','Wood'],
    ['NA','Po','Fa','TA','Gd','Ex'],
    ['NA','Po','Fa','TA','Gd','Ex'],
    ['NA','No','Mn','Av','Gd'],
    ['NA','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    ['NA','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    ['Floor','GasA','GasW','Grav','OthW','Wall'],
    ['Po','Fa','TA','Gd','Ex'],
    ['N','Y'],
    ['SBrkr','FuseA','FuseF','FuseP','Mix'],
    ['Po','Fa','TA','Gd','Ex'],
    ['NA','Po','Fa','TA','Gd','Ex'],
    ['NA','2Types','Attchd','Basment','BuiltIn','CarPort','Detchd'],
    ['NA','Unf','RFn','Fin'],
    ['NA','Po','Fa','TA','Gd','Ex'],
    ['NA','Po','Fa','TA','Gd','Ex'],
    ['NA','Fa','TA','Gd','Ex'],
    ['NA','MnWw','GdWo','MnPrv','GdPrv'],
    ['NA','Elev','Gar2','Othr','Shed','TenC'],
    ['WD','CWD','VWD','New','COD','Con','ConLw','ConLI','ConLD','Oth'],
    ['Normal','Abnorml','AdjLand','Alloca','Family','Partial']
], handle_unknown='use_encoded_value', unknown_value=-1)

data[encode_cols] = ordinal_encoder.fit_transform(data[encode_cols])
test_data[encode_cols] = ordinal_encoder.fit_transform(test_data[encode_cols])

cols_to_drop = ['Utilities','Functional','PavedDrive']

data = data.drop(columns=cols_to_drop)
test_data = test_data.drop(columns=cols_to_drop)
print(data)

In [ ]:
correlation_matrix = data.corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
#One Hot Encoding by combining and then splitting train and test data

combined_data = pd.concat([data, test_data], axis=0, ignore_index=True)
combined_encoded = pd.get_dummies(combined_data, drop_first=False)

# Split back into train and test
data = combined_encoded[combined_encoded['Id'] <= 1460].reset_index()
test_data = combined_encoded[combined_encoded['Id'] >= 1461].reset_index()
data = data.drop(['Id'], axis=1)
data.head()

In [ ]:
corr_matrix = data.corr()
# Select upper triangle of correlation matrix (to avoid duplicate pairs)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns with correlation > threshold (0.9)
high_corr_pairs = [(column, row) for column in upper.columns for row in upper.index if upper.loc[row, column] > 0.9]
to_drop=[]

# Check high-correlation pairs
for col1, col2 in high_corr_pairs:
    to_drop.append(col2)
    print(f"{col1} and {col2} have high correlation of {corr_matrix.loc[col1, col2]:.2f}")

In [ ]:
# Drop one of each highly correlated pair
data = data.drop(columns=to_drop)
data.head()

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
y = data['SalePrice']
X = data.drop('SalePrice', axis=1)
model.fit(X, y)

# Get feature importances
importances = model.feature_importances_
feature_names = X.columns

# Create a DataFrame to display top N important features
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)
print(importance_df.head(20))

# Plot top 20 features
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'][:20][::-1], importance_df['Importance'][:20][::-1])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.show()

In [ ]:
# Validating feature importance using Lasso

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit Lasso with cross-validation
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)

#Get coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lasso.coef_
})

# Filter features with non-zero coefficients
important_features = coef_df[coef_df['Coefficient'] != 0]

# Sort by absolute coefficient value (importance)
important_features = important_features.reindex(
    important_features['Coefficient'].abs().sort_values(ascending=False).index
)

# Display the top N important features (optional)
print("Important Features Selected by Lasso:\n")
print(important_features.head(20))  # Change 20 to display more or all

In [ ]:
# Plot top 20 features
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'][:20][::-1], importance_df['Importance'][:20][::-1])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.show()

plt.figure(figsize=(10, 8))
plt.barh(important_features['Feature'][:20][::-1], importance_df['Importance'][:20][::-1])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Lasso)')
plt.show()

In [ ]:
# Reduce X to only important features
imp40 = list(set(importance_df['Feature'][:30][::-1].tolist() + important_features['Feature'][:30][::-1].tolist()))
X_reduced = X[imp40]
print(X_reduced)
# X_reduced = X[important_features['Feature']]

In [ ]:
scaler = StandardScaler()
X_reduced = scaler.fit_transform(X_reduced)
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.25, random_state=42)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Define parameter grid
xgb_param_grid = {
    'n_estimators': [250, 500,1000],
    'learning_rate': [0.05, 0.07, 0.1],
    'max_depth': [3,5,6],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Initialize model
xgb_model = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')

# Grid search
xgb_grid = GridSearchCV(estimator=xgb_model, param_grid=xgb_param_grid,
                        cv=3, n_jobs=-1, scoring='r2', verbose=1)
xgb_grid.fit(X_train, y_train)

# Best model and evaluation
best_xgb = xgb_grid.best_estimator_
print("Best XGBoost Parameters:", xgb_grid.best_params_)
print("R² Score (XGBoost):", best_xgb.score(X_test, y_test))

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.07,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=1.0,
    random_state=42,
    enable_categorical=True
)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
print("Model Evaluation (XGBoost):")
print("R² Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))

In [ ]:
Ids = test_data['Id']
test_data = test_data.drop(['Id'], axis=1)
test_data.head()

In [ ]:
# test_data = test_data[important_features['Feature']]
test_data = test_data[imp40]
ytest = xgb_model.predict(test_data)
submission = pd.DataFrame({'Id': Ids, 'SalePrice': ytest})
print(submission)

In [ ]:
submission.to_csv('submission.csv', index=False)
print("✅ Submission file saved as 'submission.csv'")